# Solar Panel Calculator for Mongolia

**Question:** How much electricity can a solar panel system make at my location in Mongolia, how much money does it save, and how many years until it pays for itself?

**Data (real, free):**
1. **Open-Meteo Historical Weather API** (ERA5 reanalysis): hourly sunlight (global, direct and diffuse) and air temperature for 22 places: Ulaanbaatar and the 21 aimag centers. No API key needed. https://open-meteo.com
2. **A table of locations** that we create (name, latitude, longitude).
3. **Cost and tariff numbers** that you must look up and cite (they are settings in section 8).

**How it works:** we turn sunlight on a flat surface into sunlight on a **tilted panel** (with the `pvlib` library), adjust for temperature, add up the energy per month, and then calculate money: payback, NPV and IRR.

**How to run:** you need internet the first time (about 5 to 10 minutes for the download). The raw data is saved in a folder called `raw_solar`. The notebook also saves two small files, `solar_monthly.csv` and `locations.csv`, that the Streamlit website uses.

In [ ]:
%pip install requests pandas numpy matplotlib pvlib

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pvlib import solarposition, irradiance

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (11, 5)

## 1. Settings and the table of locations
The coordinates are approximate (the centre of each aimag center town). Check them on a map if you want to be exact. The weather data has a resolution of about 30 km, so a small difference does not matter.

In [ ]:
# ---------------- SETTINGS ----------------
THIS_YEAR = pd.Timestamp.today().year
YEARS = list(range(THIS_YEAR - 5, THIS_YEAR))      # the 5 last full years
TILTS = [0, 15, 30, 45, 60, 75, 90]                # panel tilt in degrees (0 = flat, 90 = vertical)
AZIMUTH = 180                                      # 180 = facing south (best in the northern hemisphere)
ALBEDO = 0.20                                      # ground reflection
TEMP_COEFF = -0.004                                # power change per degree C above 25 C (silicon panels)
CELL_HEAT = 0.03                                   # panel heats about 0.03 C per W/m2 of sunlight
FORCE_DOWNLOAD = False                             # True = download again even if saved files exist
# ------------------------------------------

LOCATIONS = pd.DataFrame([
    ("Ulaanbaatar", "Ulaanbaatar (capital)", 47.92, 106.92),
    ("Zuunmod", "Tov", 47.71, 106.95),
    ("Darkhan", "Darkhan-Uul", 49.49, 105.92),
    ("Erdenet", "Orkhon", 49.03, 104.04),
    ("Sukhbaatar", "Selenge", 50.24, 106.21),
    ("Bulgan", "Bulgan", 48.81, 103.53),
    ("Tsetserleg", "Arkhangai", 47.47, 101.45),
    ("Moron", "Khovsgol", 49.64, 100.16),
    ("Uliastai", "Zavkhan", 47.74, 96.85),
    ("Ulaangom", "Uvs", 49.98, 92.07),
    ("Khovd", "Khovd", 48.01, 91.64),
    ("Olgii", "Bayan-Olgii", 48.97, 89.96),
    ("Altai", "Govi-Altai", 46.37, 96.25),
    ("Bayankhongor", "Bayankhongor", 46.19, 100.72),
    ("Arvaikheer", "Uvurkhangai", 46.26, 102.78),
    ("Mandalgovi", "Dundgovi", 45.76, 106.27),
    ("Dalanzadgad", "Umnugovi", 43.57, 104.42),
    ("Sainshand", "Dornogovi", 44.89, 110.14),
    ("Choir", "Govisumber", 46.37, 108.36),
    ("Ondorkhaan", "Khentii", 47.32, 110.65),
    ("Choibalsan", "Dornod", 48.07, 114.53),
    ("Baruun-Urt", "Sukhbaatar", 46.68, 113.28),
], columns=["location", "aimag", "lat", "lon"])

LOCATIONS.to_csv("locations.csv", index=False)
print("Locations:", len(LOCATIONS), "| years:", YEARS[0], "to", YEARS[-1])
LOCATIONS.head()

## 2. Download the sunlight and temperature data
One request for each place and year (about 110 requests). The files are saved in `raw_solar`, so next time nothing is downloaded.
Open-Meteo gives the radiation as the **average of the preceding hour**, in W/m². Times are in UTC.

In [ ]:
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
VARS = ["shortwave_radiation", "direct_normal_irradiance", "diffuse_radiation", "temperature_2m"]
os.makedirs("raw_solar", exist_ok=True)


def raw_path(name):
    return os.path.join("raw_solar", name.replace(" ", "_") + ".csv.gz")


def download_location(name, lat, lon):
    path = raw_path(name)
    if os.path.exists(path) and not FORCE_DOWNLOAD:
        return pd.read_csv(path)
    frames = []
    for year in YEARS:
        params = {"latitude": lat, "longitude": lon, "start_date": f"{year}-01-01",
                  "end_date": f"{year}-12-31", "hourly": ",".join(VARS)}
        r = None
        for attempt in range(5):
            r = requests.get(ARCHIVE_URL, params=params, timeout=120)
            if r.status_code == 200:
                break
            time.sleep(60 if r.status_code == 429 else 5 * (attempt + 1))   # 429 = too many requests
        if r.status_code != 200:
            raise RuntimeError(f"Download failed ({r.status_code}) for {name} {year}: {r.text[:200]}")
        frames.append(pd.DataFrame(r.json()["hourly"]))
        time.sleep(0.5)
    df = pd.concat(frames, ignore_index=True)
    df.to_csv(path, index=False)
    return df


raw = {}
for i, row in LOCATIONS.iterrows():
    cached = os.path.exists(raw_path(row["location"])) and not FORCE_DOWNLOAD
    raw[row["location"]] = download_location(row["location"], row["lat"], row["lon"])
    print(f"{i + 1:2d}/{len(LOCATIONS)} {row['location']:<14} rows: {len(raw[row['location']]):,}", "(saved file)" if cached else "(downloaded)")

## 3. Look at the data (Ulaanbaatar)

In [ ]:
ub = raw["Ulaanbaatar"]
print(ub.shape)
print()
print("Missing values:")
print(ub.isna().sum())
print()
ub.describe().round(1)

## 4. Clean the data
- turn the time text into real dates (UTC)
- remove duplicate hours and rows with missing sunlight
- negative sunlight becomes 0 (night-time noise)
- fill short gaps (up to 3 hours) in the temperature

In [ ]:
def clean(df):
    d = df.copy()
    d["time"] = pd.to_datetime(d["time"])
    d = d.drop_duplicates(subset="time").sort_values("time")
    for c in ["shortwave_radiation", "direct_normal_irradiance", "diffuse_radiation"]:
        d.loc[d[c] < 0, c] = 0
    d["temperature_2m"] = d["temperature_2m"].interpolate(limit=3)
    d = d.dropna(subset=["shortwave_radiation", "direct_normal_irradiance",
                         "diffuse_radiation", "temperature_2m"])
    return d.reset_index(drop=True)


before = len(raw["Ulaanbaatar"])
ub_clean = clean(raw["Ulaanbaatar"])
print("Ulaanbaatar rows before:", before, "| after cleaning:", len(ub_clean))
print("Date range:", ub_clean["time"].min(), "to", ub_clean["time"].max())
ub_clean.head()

## 5. Turn sunlight into energy
For each hour we do 4 steps:
1. **Sun position** (height and direction) from the time and place, using `pvlib`.
2. **Sunlight on the tilted panel** (plane-of-array, POA): direct sunlight × how directly it hits the panel + diffuse sky light + light reflected from the ground (isotropic sky model).
3. **Panel temperature:** air temperature + `CELL_HEAT` × sunlight. Cold panels work better: efficiency changes by `TEMP_COEFF` per degree above 25 C.
4. **Energy per 1 kW of panels (kWp):** `POA / 1000 × temperature factor` (kWh per hour).

We do this for 7 tilt angles and then add up by month.

In [ ]:
def process_location(df, lat, lon, tilts, profile_tilt=45):
    d = clean(df)
    # Open-Meteo gives the mean of the PRECEDING hour, so the sun position is for the middle of that hour
    mid = pd.DatetimeIndex(d["time"] - pd.Timedelta(minutes=30)).tz_localize("UTC")
    sp = solarposition.get_solarposition(mid, lat, lon)
    ghi = pd.Series(d["shortwave_radiation"].values, index=mid)
    dni = pd.Series(d["direct_normal_irradiance"].values, index=mid)
    dhi = pd.Series(d["diffuse_radiation"].values, index=mid)
    temp = pd.Series(d["temperature_2m"].values, index=mid)

    monthly, profile = [], None
    for tilt in tilts:
        poa = irradiance.get_total_irradiance(tilt, AZIMUTH, sp["apparent_zenith"], sp["azimuth"],
                                              dni, ghi, dhi, albedo=ALBEDO, model="isotropic")["poa_global"]
        poa = poa.fillna(0).clip(lower=0)
        cell_temp = temp + CELL_HEAT * poa
        temp_factor = (1 + TEMP_COEFF * (cell_temp - 25)).clip(0.7, 1.25)
        energy = poa / 1000 * temp_factor                 # kWh per kWp in each hour

        stamp = pd.DatetimeIndex(d["time"])          # group by the hour label, so every month is complete
        t = pd.DataFrame({"year": stamp.year, "month": stamp.month, "poa": poa.values / 1000,
                          "energy": energy.values, "temp": temp.values})
        m = t.groupby(["year", "month"]).agg(poa_kwh_m2=("poa", "sum"), kwh_per_kwp=("energy", "sum"),
                                              mean_temp_c=("temp", "mean")).reset_index()
        m["tilt"] = tilt
        monthly.append(m)

        if tilt == profile_tilt:
            local = mid.tz_convert("Asia/Ulaanbaatar")
            profile = (pd.DataFrame({"month": local.month, "hour": local.hour, "energy": energy.values})
                       .groupby(["month", "hour"])["energy"].mean().reset_index())
    return pd.concat(monthly, ignore_index=True), profile


test_m, test_p = process_location(raw["Ulaanbaatar"], 47.92, 106.92, [0, 45, 90])
test_m[test_m["year"] == test_m["year"].min()].head(6).round(2)

In [ ]:
all_monthly, profiles = [], {}
for i, row in LOCATIONS.iterrows():
    m, p = process_location(raw[row["location"]], row["lat"], row["lon"], TILTS)
    m.insert(0, "location", row["location"])
    all_monthly.append(m)
    profiles[row["location"]] = p
    print(f"{i + 1:2d}/{len(LOCATIONS)} {row['location']}")

monthly = pd.concat(all_monthly, ignore_index=True)
monthly = monthly.merge(LOCATIONS, on="location")
monthly = monthly[["location", "aimag", "lat", "lon", "year", "month", "tilt",
                   "poa_kwh_m2", "kwh_per_kwp", "mean_temp_c"]].round(3)
monthly.to_csv("solar_monthly.csv", index=False)
print("\nSaved solar_monthly.csv:", monthly.shape)
monthly.head()

## 6. Key numbers
`kWh per kWp per year` means: how much energy 1 kW of panels makes in one year, before the performance ratio and snow losses of section 8.

In [ ]:
# only years with all 12 months
full = monthly.groupby(["location", "year", "tilt"])["month"].transform("count") == 12
annual = (monthly[full].groupby(["location", "year", "tilt"])["kwh_per_kwp"].sum().reset_index())
by_tilt = annual.groupby(["location", "tilt"])["kwh_per_kwp"].mean().reset_index()

best = by_tilt.loc[by_tilt.groupby("location")["kwh_per_kwp"].idxmax()].rename(
    columns={"tilt": "best_tilt", "kwh_per_kwp": "yield_best"}).merge(LOCATIONS, on="location")
best = best.sort_values("yield_best", ascending=False).reset_index(drop=True)
best.index = best.index + 1
print("Locations ranked by yearly energy per kWp at the best tilt:")
best[["location", "aimag", "best_tilt", "yield_best"]].round(0)

In [ ]:
ub_year = by_tilt[by_tilt["location"] == "Ulaanbaatar"].set_index("tilt")["kwh_per_kwp"]
print("Ulaanbaatar, kWh per kWp per year by tilt:")
print(ub_year.round(0).to_string())
print()
print("Best place: %s (%.0f kWh/kWp) | weakest place: %s (%.0f kWh/kWp)" % (
    best.iloc[0]["location"], best.iloc[0]["yield_best"], best.iloc[-1]["location"], best.iloc[-1]["yield_best"]))

## 7. Charts

### Chart 1: Map of yearly solar energy

In [ ]:
plt.figure(figsize=(12, 6))
sc = plt.scatter(best["lon"], best["lat"], c=best["yield_best"], s=260, cmap="YlOrRd", edgecolor="black")
for _, r in best.iterrows():
    plt.annotate(r["location"], (r["lon"], r["lat"]), xytext=(6, 6), textcoords="offset points", fontsize=8)
plt.colorbar(sc, label="kWh per kWp per year (best tilt)")
plt.title("Yearly solar energy by location (each dot is an aimag center)")
plt.xlabel("Longitude"); plt.ylabel("Latitude"); plt.grid(alpha=0.3); plt.show()

### Chart 2: Ranking of the locations

In [ ]:
b = best.sort_values("yield_best")
plt.figure(figsize=(9, 8))
plt.barh(b["location"], b["yield_best"], color="orange")
plt.title("Yearly energy per kWp at the best tilt")
plt.xlabel("kWh per kWp per year"); plt.grid(axis="x", alpha=0.3); plt.show()

### Chart 3: Which tilt is best? (yearly energy by tilt)

In [ ]:
plt.figure(figsize=(9, 5))
for loc in ["Ulaanbaatar", "Dalanzadgad", "Ulaangom", "Choibalsan"]:
    s = by_tilt[by_tilt["location"] == loc].set_index("tilt")["kwh_per_kwp"]
    plt.plot(s.index, s.values, marker="o", label=loc)
plt.title("Yearly energy per kWp by panel tilt (facing south)")
plt.xlabel("Tilt (0 = flat, 90 = vertical)"); plt.ylabel("kWh per kWp per year")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Chart 4: Energy by month for one location

In [ ]:
# ---------------- SETTINGS ----------------
LOCATION = "Ulaanbaatar"
TILT = 45                 # one of 0, 15, 30, 45, 60, 75, 90
# ------------------------------------------

one = monthly[(monthly["location"] == LOCATION) & (monthly["tilt"] == TILT)]
avg_month = one.groupby("month")["kwh_per_kwp"].mean()

plt.bar(avg_month.index, avg_month.values, color="gold", edgecolor="black")
plt.xticks(range(1, 13))
plt.title(f"{LOCATION}: average energy per kWp by month (tilt {TILT} degrees)")
plt.xlabel("Month"); plt.ylabel("kWh per kWp"); plt.grid(axis="y", alpha=0.3); plt.show()
print("Best month: %d (%.0f kWh) | worst month: %d (%.0f kWh)" % (
    avg_month.idxmax(), avg_month.max(), avg_month.idxmin(), avg_month.min()))

### Chart 5: Energy by month and hour of the day (local time)

In [ ]:
_, prof = process_location(raw[LOCATION], LOCATIONS.set_index("location").loc[LOCATION, "lat"],
                           LOCATIONS.set_index("location").loc[LOCATION, "lon"], [TILT], profile_tilt=TILT)
pivot = prof.pivot(index="month", columns="hour", values="energy")

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels(range(0, 24, 2))
ax.set_yticks(range(12)); ax.set_yticklabels(range(1, 13))
ax.set_xlabel("Hour of the day (local time)"); ax.set_ylabel("Month")
ax.set_title(f"{LOCATION}: average energy per kWp by month and hour (kWh)")
fig.colorbar(im, label="kWh per kWp per hour")
plt.tight_layout(); plt.show()

### Chart 6: How much does the energy change from year to year?

In [ ]:
yr = annual[(annual["location"] == LOCATION) & (annual["tilt"] == TILT)].set_index("year")["kwh_per_kwp"]
plt.bar(yr.index.astype(str), yr.values, color="seagreen")
plt.axhline(yr.mean(), color="black", linestyle="--", label="Average")
plt.title(f"{LOCATION}: yearly energy per kWp (tilt {TILT})")
plt.ylabel("kWh per kWp"); plt.legend(); plt.grid(axis="y", alpha=0.3); plt.show()
print("Difference between best and worst year: %.1f%%" % ((yr.max() - yr.min()) / yr.mean() * 100))

## 8. The money: payback, NPV and IRR
**Important:** the cost and tariff values below are **placeholders**. Replace them with real, cited values:
- electricity tariff (MNT per kWh): Energy Regulatory Commission of Mongolia
- cost of a system (MNT per kW): a quote from a local installer, or a published report
Use the same units for everything. All other values are assumptions you can change.

How the calculation works:
- Energy in year 1 = sum over months of (kWh per kWp × system size × performance ratio × (1 - snow loss in winter months)).
- Each year the panels lose a little power (degradation). The tariff grows by `TARIFF_GROWTH` per year.
- Savings = energy used × tariff. The yearly cost of maintenance is a share of the system cost.
- **Payback** = the year when the total cash flow becomes positive. **NPV** = value today of all future cash flows. **IRR** = the yearly return of the investment.

In [ ]:
# ---------------- SETTINGS (replace the placeholders) ----------------
LOCATION = "Ulaanbaatar"
TILT = 45
SYSTEM_KW = 3                  # system size in kW
COST_PER_KW = 3_500_000        # PLACEHOLDER: MNT per kW of panels, installed
TARIFF = 250                   # PLACEHOLDER: MNT per kWh
TARIFF_GROWTH = 0.05           # PLACEHOLDER: yearly growth of the tariff
PERFORMANCE_RATIO = 0.80       # losses from cables, dust, inverter (typical 0.75-0.85)
SNOW_LOSS = 0.05               # share of energy lost to snow in Nov-Mar
DEGRADATION = 0.005            # yearly loss of panel power
MAINTENANCE = 0.01             # yearly maintenance as a share of the system cost
SELF_USE = 1.0                 # share of the energy that you use yourself (1 = all)
DISCOUNT_RATE = 0.10           # yearly discount rate for the NPV
LIFETIME = 25                  # years
# ----------------------------------------------------------------------

SNOW_MONTHS = [11, 12, 1, 2, 3]


def year1_energy(loc, tilt, kw, pr, snow):
    m = monthly[(monthly["location"] == loc) & (monthly["tilt"] == tilt)]
    avg = m.groupby("month")["kwh_per_kwp"].mean()
    factor = pd.Series({mo: (1 - snow if mo in SNOW_MONTHS else 1.0) for mo in avg.index})
    return float((avg * factor).sum() * kw * pr)


def cash_flows(e1, cost, tariff, growth, degr, maint, self_use, years):
    flows = [-cost]
    for t in range(1, years + 1):
        energy = e1 * (1 - degr) ** (t - 1)
        saving = energy * self_use * tariff * (1 + growth) ** (t - 1)
        flows.append(saving - cost * maint)
    return np.array(flows)


def payback_years(flows):
    cum = np.cumsum(flows)
    if not (cum >= 0).any():
        return np.nan
    i = int(np.argmax(cum >= 0))
    return (i - 1) + (-cum[i - 1] / flows[i]) if i > 0 else 0.0


def npv(rate, flows):
    return float(sum(f / (1 + rate) ** t for t, f in enumerate(flows)))


def irr(flows):
    lo, hi = -0.9, 5.0
    if npv(lo, flows) * npv(hi, flows) > 0:
        return np.nan
    for _ in range(200):
        mid = (lo + hi) / 2
        if npv(lo, flows) * npv(mid, flows) <= 0:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2


cost = SYSTEM_KW * COST_PER_KW
e1 = year1_energy(LOCATION, TILT, SYSTEM_KW, PERFORMANCE_RATIO, SNOW_LOSS)
flows = cash_flows(e1, cost, TARIFF, TARIFF_GROWTH, DEGRADATION, MAINTENANCE, SELF_USE, LIFETIME)

print(f"Location: {LOCATION} | system: {SYSTEM_KW} kW | tilt {TILT} degrees")
print(f"Energy in year 1:   {e1:,.0f} kWh")
print(f"System cost:        {cost:,.0f} MNT")
print(f"Savings in year 1:  {flows[1] + cost * MAINTENANCE:,.0f} MNT (before maintenance)")
print(f"Payback:            {payback_years(flows):.1f} years")
print(f"NPV ({LIFETIME} years):    {npv(DISCOUNT_RATE, flows):,.0f} MNT")
print(f"IRR:                {irr(flows) * 100:.1f}% per year")

In [ ]:
cum = np.cumsum(flows)
pb = payback_years(flows)

plt.figure(figsize=(10, 5))
plt.plot(range(len(cum)), cum / 1e6, color="seagreen", linewidth=2)
plt.axhline(0, color="black", linewidth=1)
if not np.isnan(pb):
    plt.axvline(pb, color="red", linestyle="--", label=f"Payback: {pb:.1f} years")
plt.title("Cumulative cash flow of the solar system")
plt.xlabel("Year"); plt.ylabel("Million MNT"); plt.legend(); plt.grid(alpha=0.3); plt.show()

### Payback in every location (same system and settings)

In [ ]:
rows = []
for loc in LOCATIONS["location"]:
    e = year1_energy(loc, TILT, SYSTEM_KW, PERFORMANCE_RATIO, SNOW_LOSS)
    f = cash_flows(e, cost, TARIFF, TARIFF_GROWTH, DEGRADATION, MAINTENANCE, SELF_USE, LIFETIME)
    rows.append({"location": loc, "energy_kwh": round(e), "payback_years": round(payback_years(f), 1),
                 "npv_mnt": round(npv(DISCOUNT_RATE, f))})
loc_table = pd.DataFrame(rows).sort_values("payback_years").reset_index(drop=True)

plt.figure(figsize=(9, 8))
t = loc_table.sort_values("payback_years", ascending=False)
plt.barh(t["location"], t["payback_years"], color="steelblue")
plt.title(f"Payback time by location ({SYSTEM_KW} kW, tilt {TILT})")
plt.xlabel("Years"); plt.grid(axis="x", alpha=0.3); plt.show()
loc_table.head(10)

### What if the tariff or the cost is different? (sensitivity)
The tariff and the cost are the least certain numbers, so we test other values. Each cell is the payback in years.

In [ ]:
tariff_mult = [0.6, 0.8, 1.0, 1.2, 1.5]
cost_mult = [0.6, 0.8, 1.0, 1.2, 1.5]
grid = pd.DataFrame(index=[f"{int(c*100)}%" for c in cost_mult], columns=[f"{int(t*100)}%" for t in tariff_mult], dtype=float)
for c in cost_mult:
    for t in tariff_mult:
        f = cash_flows(e1, cost * c, TARIFF * t, TARIFF_GROWTH, DEGRADATION, MAINTENANCE, SELF_USE, LIFETIME)
        grid.loc[f"{int(c*100)}%", f"{int(t*100)}%"] = payback_years(f)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(grid.values.astype(float), cmap="RdYlGn_r", aspect="auto")
ax.set_xticks(range(len(tariff_mult))); ax.set_xticklabels(grid.columns)
ax.set_yticks(range(len(cost_mult))); ax.set_yticklabels(grid.index)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        v = grid.values[i, j]
        ax.text(j, i, "never" if np.isnan(v) else f"{v:.1f}", ha="center", va="center", fontsize=9)
ax.set_xlabel("Tariff (compared with your setting)"); ax.set_ylabel("System cost (compared with your setting)")
ax.set_title("Payback in years")
fig.colorbar(im, label="Years"); plt.tight_layout(); plt.show()

## 9. Findings
*Replace the text below with your own results after you run the notebook.*

1. The best location is ___ with ___ kWh per kWp per year, and the weakest is ___ with ___ kWh. The difference is ___%.
2. The best tilt is about ___ degrees. A flat panel (0 degrees) makes ___% less energy than the best tilt in Ulaanbaatar. A vertical panel makes ___% less.
3. Energy in the best month (___) is ___ times the energy in the worst month (___).
4. The energy changes by about ___% between the best and worst year, so a single year is a good estimate (or not).
5. For a ___ kW system in ___ with my settings: ___ kWh in year 1, payback ___ years, NPV ___ MNT, IRR ___%.
6. The payback changes from ___ to ___ years when the tariff goes from 60% to 150% of my setting, so the ___ matters most.

## 10. About the data and limits
- **Source:** Open-Meteo Historical Weather API (ERA5 reanalysis, Copernicus / ECMWF), hourly shortwave radiation, direct normal irradiance, diffuse radiation and air temperature. https://open-meteo.com/en/docs/historical-weather-api
- **License:** free for non-commercial use under CC BY 4.0. Please credit Open-Meteo.
- **Locations:** 22 places (Ulaanbaatar and the 21 aimag centers), approximate coordinates.
- **Method:** sun position and sunlight on a tilted panel with `pvlib` (isotropic sky model, ground reflection 0.20), panel temperature effect, then monthly sums.
- **Cleaning:** parsed dates, removed duplicate hours and missing values, set negative sunlight to 0, filled short temperature gaps.
- **Limits:**
  - ERA5 sunlight is a model estimate on a grid of about 30 km. It can differ from a real measurement on your roof.
  - Ground reflection is fixed at 0.20, but snow reflects more, so winter energy may be a little higher in reality.
  - Snow cover on the panels is only a simple percentage loss. Shade, dust and the real inverter are not modeled.
  - Cost and tariff values are placeholders until you replace them with cited values.
  - The model assumes you use the energy yourself. Rules for selling extra electricity to the grid are not included.
  - The panels face south only.